# 00 — Extract vanilla CLIP image embeddings for items_phase_2

The baseline siamese model (`artifacts/models/siamese_baseline.pth`, F1≈0.85
end-to-end) was trained against **vanilla CLIP-ViT-B/32** image embeddings
(768-d). The existing `artifacts/embeddings/clip_image.pt` only covers
items_train + items_phase_1 (1,128,069 items).

When `phase1/05_phase2_baseline.ipynb` runs on `items_phase_2.csv` it can't
find embeddings for those 197,837 items → silent zero vectors → broken
clustering.

This notebook mirrors the original extraction recipe (same `openai/clip-vit-base-patch32`,
same 768-d output) but processes **only the missing phase_2 ids** and
appends them in place. Resume-safe.

**Cost**: ~few hours on M4 MPS at batch 128 for 197k images.


In [1]:
import sys, os
sys.path.insert(0, '/Users/matouskovar/FIT/adm-sp/src')

REPO_ROOT       = '/Users/matouskovar/FIT/adm-sp'
DATA_DIR        = f'{REPO_ROOT}/data'
ARTIFACTS_DIR   = f'{REPO_ROOT}/artifacts'
CLIP_EMB_PATH   = f'{ARTIFACTS_DIR}/embeddings/clip_image.pt'   # the file we're appending to
IMAGES_DIR      = '/Users/matouskovar/FIT/images'
MODEL_ID        = 'openai/clip-vit-base-patch32'   # MUST match what trained the baseline siamese
BATCH_SIZE      = 128
CHECKPOINT_EVERY = 50_000

import torch
device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: mps


In [2]:
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPProcessor, CLIPVisionModel

# Items we want to add: just items_phase_2
df_p2 = pd.read_csv(f'{DATA_DIR}/items_phase_2.csv', usecols=['itemId'])
p2_ids = set(df_p2['itemId'].astype(str))
print(f'items_phase_2: {len(p2_ids):,}')

# Load existing dict and figure out what's missing
if os.path.exists(CLIP_EMB_PATH):
    embs = torch.load(CLIP_EMB_PATH, map_location='cpu', weights_only=False)
    print(f'Existing clip_image.pt: {len(embs):,} entries')
else:
    embs = {}
    print('No existing clip_image.pt — starting from scratch.')

remaining = sorted(p2_ids - set(embs.keys()))
print(f'Phase 2 items still to embed: {len(remaining):,}')


items_phase_2: 197,837
Existing clip_image.pt: 1,128,069 entries
Phase 2 items still to embed: 197,837


In [3]:
print(f'Loading {MODEL_ID}...')
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model     = CLIPVisionModel.from_pretrained(MODEL_ID).to(device).eval()
print('  Vision hidden size:', model.config.hidden_size)   # 768 for ViT-B/32


Loading openai/clip-vit-base-patch32...


The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias          

  Vision hidden size: 768


In [4]:
@torch.no_grad()
def embed_batch(item_ids):
    imgs, kept = [], []
    for iid in item_ids:
        try:
            img = Image.open(f'{IMAGES_DIR}/{iid}.jpg').convert('RGB')
            imgs.append(img); kept.append(iid)
        except Exception:
            pass   # missing/corrupt -> silently skip; will be a singleton at clustering
    if not imgs:
        return {}
    inputs = processor(images=imgs, return_tensors='pt').to(device)
    feats  = model(**inputs).pooler_output.cpu()    # (B, 768)
    return {iid: feats[i] for i, iid in enumerate(kept)}

processed_since_save = 0
for start in tqdm(range(0, len(remaining), BATCH_SIZE), desc='Embedding phase_2'):
    batch = remaining[start : start + BATCH_SIZE]
    embs.update(embed_batch(batch))
    processed_since_save += len(batch)
    if processed_since_save >= CHECKPOINT_EVERY:
        torch.save(embs, CLIP_EMB_PATH)
        processed_since_save = 0

torch.save(embs, CLIP_EMB_PATH)
print(f'\nSaved {len(embs):,} entries → {CLIP_EMB_PATH}')


Embedding phase_2:   0%|          | 0/1546 [00:00<?, ?it/s]


Saved 1,325,906 entries → /Users/matouskovar/FIT/adm-sp/artifacts/embeddings/clip_image.pt


## Verify coverage

In [5]:
embs_check = torch.load(CLIP_EMB_PATH, map_location='cpu', weights_only=False)
covered = sum(1 for iid in p2_ids if iid in embs_check)
sample = next(iter(embs_check.values()))
print(f'phase_2 coverage : {covered:,} / {len(p2_ids):,}')
print(f'sample dim       : {sample.shape}')
print(f'total entries    : {len(embs_check):,}')

# Expected: covered == 197,837 (or just below if some images are missing on disk)
# Expected total: 1,128,069 (existing) + ~197,837 (new) ≈ 1,325,906


phase_2 coverage : 197,837 / 197,837
sample dim       : torch.Size([768])
total entries    : 1,325,906
